In [0]:
%run ../FASE1/00_utils

In [0]:
%run ./00_utility

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# 1. Dataset Preparation

In [0]:
df_dettagli = spark.table('ta_prod.silver.v_trasmesso_ta').drop(*COLUMNS_TO_REMOVE_V_TRASMESSO_TA).distinct()
df_dettagli = df_dettagli.groupBy([col for col in df_dettagli.columns if col != 'ETA_MEDIA']).agg(F.mean("ETA_MEDIA").alias("ETA_MEDIA"))

df_dettagli = df_dettagli.withColumnsRenamed({'DATA': 'Data', 'DES_TITOLO_PRINC_INT': 'Programma', 'DES_NETWORK': 'Canale'})
df_dettagli = convert_time_to_seconds(df_dettagli, ['ORA_INIZIO_TRX', 'ORA_FINE_TRX'])

for column in ['Programma','Canale','DES_GENERE_ESTESA_INT','DES_GENERE_FILM_INT','DES_GENERE_SPORT_INT','DES_MANIFESTAZIONE_SPORT_INT','DES_SPECIALITA_SPORT_INT']:
    df_dettagli = df_dettagli.withColumn(column, F.trim(column))

# format channel names
df_dettagli = df_dettagli.withColumn(
    "Canale",
    F.when(F.col("Canale") == "Italia1", "Italia 1")
    .when(F.col("Canale") == "Canale5", "Canale 5")
    .when(F.col("Canale") == "Dmax Tv", "Dmax")
    .when(F.col("Canale") == "FRISBEE", "Frisbee")
    .when(F.col("Canale") == "Italia 2", "Italia 2 Mediaset")
    .when(F.col("Canale") == "K2 (nazionale)", "K2")
    .when(F.col("Canale") == "Rai GULP", "Rai Gulp")
    .when(F.col("Canale") == "Rai YoYo", "Rai Yoyo")
    .when(F.col("Canale") == "Rete4", "Rete 4")
    .when(F.col("Canale") == "Rai YoYo", "Rai Yoyo")
    .when(F.col("Canale") == "Sky TG24", "Sky Tg24")
    .when(F.col("Canale") == "TV8", "Tv8")
    .otherwise(F.col('Canale'))
)

df_dettagli.limit(100).display()

In [0]:
df_programmi = spark.table('ta_prod.gold.v_report_programmi')

df_programmi = convert_time_to_seconds(df_programmi, ['Minuto', 'ORA_INIZIO_TRX', 'ORA_FINE_TRX'])
df_programmi = df_programmi.drop(*set(COLUMNS_TO_REMOVE + FIRST_SREEN_LIVE_VOSDAL_REGIONI + VOD_COLUMNS))

df_programmi = df_programmi.dropna(subset=['LiveVOSDAL','ORA_INIZIO_TRX','ORA_FINE_TRX'])
df_programmi = groupby_single_row(df_programmi)

df_programmi = remove_contenitori(df_programmi)
# df_programmi = compute_total_share(df_programmi)
df_programmi = compute_audience_category_percentage(df_programmi)

df_programmi = df_programmi.join(df_dettagli, ['Data', 'Canale', 'ORA_INIZIO_TRX', 'ORA_FINE_TRX', 'Programma'], 'left')

df_programmi = df_programmi.dropna(subset=['Share'])
df_programmi = df_programmi.dropDuplicates(['Data', 'Canale', 'Programma', 'ORA_INIZIO_TRX'])
df_programmi = df_programmi.withColumn('ID', F.concat(F.col('Canale'), F.lit('_'), F.col('Data'), F.lit('_'), F.col('Programma'), F.lit('_'), F.col('ORA_INIZIO_TRX')))

In [0]:
# Creiamo delle udfs perchè le funzioni di normalizzazione sono definite in python
normalize_title_udf = F.udf(normalize_title, StringType())
apply_manual_mapping_udf = F.udf(apply_manual_mapping, StringType())
 
# Creiamo una nuova colonna con i nomi programma normalizzati
df_programmi = df_programmi.withColumn("programma_norm", normalize_title_udf(F.col("Programma")))
df_programmi = df_programmi.withColumn("programma_norm", apply_manual_mapping_udf(F.col("Canale"), F.col("programma_norm")))

In [0]:
# Scrittura dei dati in tabella
df_programmi.write.mode('overwrite').saveAsTable('ta_coll.whatif.storico_programmi')
# Liquid Clustering su Data, Canale e programma_norm
# Forza la raccolta delle statistiche Delta sulle colonne del clustering
spark.sql("ALTER TABLE ta_coll.whatif.storico_programmi SET TBLPROPERTIES ('delta.dataSkippingStatsColumns' = 'Data,Canale')")
spark.sql('ALTER TABLE ta_coll.whatif.storico_programmi CLUSTER BY (Data, Canale)')
spark.sql('OPTIMIZE ta_coll.whatif.storico_programmi')

In [0]:
df_programmi = spark.table('ta_coll.whatif.storico_programmi')
df_programmi.display()

In [0]:
print('# righe:', df_programmi.count())
print('# colonne:', len(df_programmi.columns))